# 面试问题：Agent 生成计划后，怎样校验 DAG、并行调度、处理失败和动态重规划？

**一句话回答**：计划不是自然语言清单，而是带 task ID、依赖、工具、输入/输出 schema、风险和预算的 DAG。宿主验证唯一 ID、依赖存在、无环和权限；用拓扑 ready set 并行执行，失败按语义重试并阻断依赖节点；重规划只能追加或替换未完成子图，已提交副作用必须通过幂等或补偿处理。

本 Notebook 手写 Kahn 拓扑排序、执行波次、critical path、失败传播与安全重规划。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
from collections import defaultdict, deque  # 导入本单元所需的依赖。
import hashlib, json, math  # 导入本单元所需的依赖。

SEED116=11601  # 计算并保存当前步骤的中间状态。
assert SEED116==11601  # 用受控断言验证关键不变量。
assert deque([1]).popleft()==1  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"plan-v1").hexdigest()!=hashlib.sha256(b"plan-v2").hexdigest()  # 用受控断言验证关键不变量。

## 1. Plan Node 是执行合同

节点声明 `id/deps/tool/args/duration/write/idempotency`。工具必须来自允许注册表，依赖只能引用计划内节点；写节点没有幂等键或补偿策略就不能自动重试。模型输出先过 schema，不能直接进入调度器。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Node116:  # 定义承载本节状态与行为的数据结构。
    node_id:str; deps:tuple; tool:str; duration:int=1; write:bool=False; idempotent:bool=True  # 计算并保存当前步骤的中间状态。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if not self.node_id or self.duration<=0 or self.node_id in self.deps: raise ValueError("node_contract")  # 按当前条件选择后续控制路径。
plan116=[Node116("search_a",(),"search",2),Node116("search_b",(),"search",3),Node116("merge",("search_a","search_b"),"merge",1),Node116("publish",("merge",),"publish",2,True,True)]  # 计算并保存当前步骤的中间状态。
assert len({n.node_id for n in plan116})==4  # 用受控断言验证关键不变量。
assert plan116[-1].write and plan116[-1].idempotent  # 用受控断言验证关键不变量。
try: Node116("x",("x",),"t"); raise AssertionError("self dependency accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="node_contract"  # 捕获预期异常并验证失败分支。

## 2. Kahn 算法同时验证依赖和环

先建立 indegree 与反向邻接，反复弹出入度为零节点；若输出数少于节点数则存在环。为了可复现，ready queue 按 ID 排序。环不是“多试一次”能解决的错误，应让 planner 重写计划。

In [ ]:
def topo116(nodes):  # 定义本节可复用的核心函数。
    ids=[n.node_id for n in nodes]  # 计算并保存当前步骤的中间状态。
    if len(ids)!=len(set(ids)): raise ValueError("duplicate_id")  # 按当前条件选择后续控制路径。
    by={n.node_id:n for n in nodes}; missing={d for n in nodes for d in n.deps if d not in by}  # 计算并保存当前步骤的中间状态。
    if missing: raise ValueError("missing_dependency")  # 按当前条件选择后续控制路径。
    indeg={i:0 for i in ids}; children=defaultdict(list)  # 计算并保存当前步骤的中间状态。
    for n in nodes:  # 遍历输入元素以累积或检查结果。
        indeg[n.node_id]=len(n.deps)  # 计算并保存当前步骤的中间状态。
        for d in n.deps: children[d].append(n.node_id)  # 遍历输入元素以累积或检查结果。
    ready=sorted([i for i,v in indeg.items() if v==0]); order=[]  # 计算并保存当前步骤的中间状态。
    while ready:  # 在终止条件满足前持续推进状态。
        x=ready.pop(0); order.append(x)  # 计算并保存当前步骤的中间状态。
        for c in sorted(children[x]):  # 遍历输入元素以累积或检查结果。
            indeg[c]-=1  # 计算并保存当前步骤的中间状态。
            if indeg[c]==0: ready.append(c); ready.sort()  # 按当前条件选择后续控制路径。
    if len(order)!=len(nodes): raise ValueError("cycle")  # 按当前条件选择后续控制路径。
    return order  # 返回当前分支计算出的结果。
assert topo116(plan116)==["search_a","search_b","merge","publish"]  # 用受控断言验证关键不变量。
try: topo116([Node116("a",("b",),"x"),Node116("b",("a",),"x")]); raise AssertionError("cycle accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="cycle"  # 捕获预期异常并验证失败分支。
try: topo116([Node116("a",("z",),"x")]); raise AssertionError("missing accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="missing_dependency"  # 捕获预期异常并验证失败分支。

## 3. Ready Set 形成可并行执行波次

同一波节点的依赖均已完成，但还要检查它们是否会写同一资源。波次只表达拓扑并行性，实际并发还受模型/工具配额、租户公平和 deadline 限制。每个结果按 node ID 归档，聚合顺序确定。

In [ ]:
def waves116(nodes):  # 定义本节可复用的核心函数。
    remaining={n.node_id:n for n in nodes}; done=set(); out=[]  # 计算并保存当前步骤的中间状态。
    while remaining:  # 在终止条件满足前持续推进状态。
        ready=sorted(i for i,n in remaining.items() if set(n.deps)<=done)  # 计算并保存当前步骤的中间状态。
        if not ready: raise ValueError("cycle")  # 按当前条件选择后续控制路径。
        out.append(ready); done.update(ready)  # 执行当前语句以推进本节示例。
        for i in ready: del remaining[i]  # 遍历输入元素以累积或检查结果。
    return out  # 返回当前分支计算出的结果。
wave116=waves116(plan116)  # 计算并保存当前步骤的中间状态。
assert wave116==[["search_a","search_b"],["merge"],["publish"]]  # 用受控断言验证关键不变量。
assert set(wave116[0]).isdisjoint(set(wave116[1]))  # 用受控断言验证关键不变量。
assert sum(map(len,wave116))==len(plan116)  # 用受控断言验证关键不变量。

## 4. Critical Path 给出理想下界

无限并发下总延迟由最长依赖路径决定，不是所有 duration 相加。对拓扑序做动态规划：`finish(node)=duration+max(finish(dep))`。真实延迟还含排队、重试、模型批处理和限流。

In [ ]:
def critical_path116(nodes):  # 定义本节可复用的核心函数。
    by={n.node_id:n for n in nodes}; finish={}; parent={}  # 计算并保存当前步骤的中间状态。
    for i in topo116(nodes):  # 遍历输入元素以累积或检查结果。
        n=by[i]; best=max(n.deps,key=lambda d:finish[d]) if n.deps else None; finish[i]=n.duration+(finish[best] if best else 0); parent[i]=best  # 计算并保存当前步骤的中间状态。
    end=max(finish,key=finish.get); path=[]  # 计算并保存当前步骤的中间状态。
    while end is not None: path.append(end); end=parent[end]  # 在终止条件满足前持续推进状态。
    return finish[max(finish,key=finish.get)],path[::-1]  # 返回当前分支计算出的结果。
critical116,path116=critical_path116(plan116)  # 计算并保存当前步骤的中间状态。
assert critical116==6  # 用受控断言验证关键不变量。
assert path116==["search_b","merge","publish"]  # 用受控断言验证关键不变量。
assert critical116<sum(n.duration for n in plan116)  # 用受控断言验证关键不变量。

## 5. 失败传播只阻断依赖后代

一个 search 失败不应抹掉已完成的独立 search；其后代标为 blocked，其他分支可继续。最终 orchestrator 根据必需/可选节点决定 partial answer、重规划或失败。下面显式计算失败闭包。

In [ ]:
def blocked_by116(nodes,failed):  # 定义本节可复用的核心函数。
    blocked=set(failed); changed=True  # 计算并保存当前步骤的中间状态。
    while changed:  # 在终止条件满足前持续推进状态。
        changed=False  # 计算并保存当前步骤的中间状态。
        for n in nodes:  # 遍历输入元素以累积或检查结果。
            if n.node_id not in blocked and set(n.deps)&blocked: blocked.add(n.node_id); changed=True  # 按当前条件选择后续控制路径。
    return blocked  # 返回当前分支计算出的结果。
blocked116=blocked_by116(plan116,{"search_a"})  # 计算并保存当前步骤的中间状态。
assert blocked116=={"search_a","merge","publish"}  # 用受控断言验证关键不变量。
assert "search_b" not in blocked116  # 用受控断言验证关键不变量。
assert blocked_by116(plan116,set())==set()  # 用受控断言验证关键不变量。

## 6. Retry 取决于错误与副作用语义

timeout/429 的只读节点可指数退避；写节点只有幂等或可查询提交状态时才自动重试。schema/权限错误应立即失败，不能通过重试绕过。重试预算属于节点也属于整个 run，防止 fan-out 放大请求。

In [ ]:
def retry_policy116(node,error,attempt,max_attempts=3):  # 定义本节可复用的核心函数。
    if error in {"schema","unauthorized"}: return "fail"  # 按当前条件选择后续控制路径。
    if node.write and not node.idempotent: return "reconcile"  # 按当前条件选择后续控制路径。
    if error in {"timeout","rate_limit"} and attempt<max_attempts: return "retry"  # 按当前条件选择后续控制路径。
    return "fail"  # 返回当前分支计算出的结果。
assert retry_policy116(plan116[0],"timeout",1)=="retry"  # 用受控断言验证关键不变量。
assert retry_policy116(Node116("w",(),"write",1,True,False),"timeout",1)=="reconcile"  # 用受控断言验证关键不变量。
assert retry_policy116(plan116[0],"unauthorized",0)=="fail"  # 用受控断言验证关键不变量。

## 7. 重规划只能修改未完成子图

planner 接收目标、completed 结果和失败原因，返回 replacement nodes。新计划不能复用不同语义的 ID，不能让新节点依赖已删除的失败节点；已完成写操作不可“忘记”，后续要么接受，要么显式补偿。

In [ ]:
completed116={"search_b":{"result":"B"}}; replacement116=[Node116("search_a_retry",(),"backup_search",2),Node116("merge_v2",("search_a_retry","search_b"),"merge",1),Node116("publish_v2",("merge_v2",),"publish",2,True,True)]  # 计算并保存当前步骤的中间状态。
combined116=[Node116("search_b",(),"completed",3)]+replacement116  # 计算并保存当前步骤的中间状态。
order_replan116=topo116(combined116)  # 计算并保存当前步骤的中间状态。
assert set(completed116)<=set(order_replan116)  # 用受控断言验证关键不变量。
assert order_replan116.index("search_b")<order_replan116.index("merge_v2")  # 用受控断言验证关键不变量。
assert "search_a" not in order_replan116 and "search_a_retry" in order_replan116  # 用受控断言验证关键不变量。

## 8. Plan 与执行状态都要版本化

保存原目标、planner/model 版本、原始 plan、校验结果、每节点输入摘要/attempt/result、重规划 diff 与最终状态。离线测试包括环、缺依赖、越权工具、部分失败、重复回调和恢复；计划质量指标包含成功率、critical-path 膨胀和无用节点比例。

In [ ]:
plan_payload116=[{"id":n.node_id,"deps":n.deps,"tool":n.tool,"duration":n.duration,"write":n.write,"idempotent":n.idempotent} for n in plan116]  # 计算并保存当前步骤的中间状态。
manifest116={"schema":1,"planner":"planner-v3","plan":plan_payload116,"topology":"dag","scheduler":"ready-set-v2","retry":"semantic","replan":"preserve-completed"}; digest116=hashlib.sha256(json.dumps(manifest116,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert len(manifest116["plan"])==4  # 用受控断言验证关键不变量。
assert manifest116["replan"]=="preserve-completed"  # 用受控断言验证关键不变量。
assert len(digest116)==64  # 用受控断言验证关键不变量。

## 面试总结

回答顺序是：**结构化节点 → schema/权限 → Kahn 无环验证 → ready waves → critical path → 局部失败传播 → 语义重试 → 保留已完成状态的重规划 → 版本化 trace**。LLM 负责提出计划，确定性 scheduler 负责保证计划不会越过执行合同。

延伸阅读：[PlanBench](https://arxiv.org/abs/2206.10498)、[Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents)、[ReAct](https://arxiv.org/abs/2210.03629)。